## STEP 1: Install Dependencies

In [27]:
!pip install -U langchain langchain-community langchain-text-splitters chromadb sentence-transformers pypdf langgraph

## STEP 2: Upload PDF

In [89]:
from google.colab import files
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]
print("Uploaded file:", pdf_name)

Saving support_kb.pdf to support_kb.pdf
Uploaded file: support_kb.pdf


## STEP 3: Load PDF

In [90]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_name)
docs = loader.load()

print("Pages loaded:", len(docs))

Pages loaded: 1


## STEP 4: Chunking

In [92]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = splitter.split_documents(docs)

print("Chunks created:", len(chunks))


Chunks created: 4


## STEP 5: Embeddings

In [93]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings()

/tmp/ipykernel_8744/314511280.py:3: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embedding = HuggingFaceEmbeddings()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## STEP 6: Store in ChromaDB

In [94]:
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(chunks, embedding)
retriever = db.as_retriever(search_kwargs={"k": 3})

print("✅ Data stored in ChromaDB")

✅ Data stored in ChromaDB


## STEP 7: Simple Answer Function


In [95]:
def get_answer(query):
    results = db.similarity_search(query, k=1)

    text = results[0].page_content
    lines = text.split("\n")

    query_clean = query.lower()

    answers = []
    capture = False

    for line in lines:
        line_clean = line.strip().lower()

        # match question
        if any(word in line_clean for word in query_clean.split()):
            capture = True
            continue

        # collect bullet answers
        if capture:
            if line.strip().startswith("-") or line.strip().startswith("*"):
                answers.append(line.replace("-", "").replace("*", "").strip())
            else:
                break

    if answers:
        return " ".join(answers)
    else:
        return results[0].page_content

## STEP 8: Build LangGraph Workflow

In [97]:
from langgraph.graph import StateGraph, END
from typing import TypedDict

# STATE
class GraphState(TypedDict):
    query: str
    answer: str
    confidence: float

#Process node
def process_node(state):
    query = state["query"]

    answer = get_answer(query)

    # simple confidence logic
    if len(answer) < 50:
        confidence = 0.3
    else:
        confidence = 0.9

    return {
        "query": query,
        "answer": answer,
        "confidence": confidence
    }

#Router Node
def router(state):
    if state["confidence"] < 0.4:
        return "hitl"
    return "output"

#HITL Node
def hitl_node(state):
    print("\n⚠️ Escalated to Human Support")
    human = input("Enter human response: ")
    state["answer"] = human
    return state

#Output Node
def output_node(state):
    print("\n🤖 Final Answer:\n", state["answer"])
    return state

#Build Graph
builder = StateGraph(GraphState)

builder.add_node("process", process_node)
builder.add_node("hitl", hitl_node)
builder.add_node("output", output_node)

builder.set_entry_point("process")

builder.add_conditional_edges(
    "process",
    router,
    {
        "hitl": "hitl",
        "output": "output"
    }
)

builder.add_edge("hitl", "output")
builder.add_edge("output", END)

app = builder.compile()

print("✅ LangGraph Ready")

✅ LangGraph Ready


## STEP 9: Run the Chatbot

In [101]:
while True:
    query = input("\n💬 Ask your question (or type 'exit'): ")

    if query.lower() == "exit":
        break

    state = {
        "query": query,
        "answer": "",
        "confidence": 0.0
    }

    app.invoke(state)


💬 Ask your question (or type 'exit'):  Billing and Payments ? 

🤖 Final Answer:
 Monthly plans are billed at the start of each cycle. Annual plans are billed upfront.

💬 Ask your question (or type 'exit'): SLAs ? 

🤖 Final Answer:
 Standard priority: first response within 8 business hours, High priority: first response within 2 business hours.

💬 Ask your question (or type 'exit'): Technical Troubleshooting ?  

🤖 Final Answer:
 Duplicate charge complaints should be checked against invoice IDs and payment timestamps. Accepted payment methods are

💬 Ask your question (or type 'exit'): Cancellation and Plan Changes ? 

🤖 Final Answer:
 Escalate incidents affecting multiple users as potential service outage. Users can cancel anytime from Billing Settings. Access continues until period end. Downgrades apply on next billing cycle. Upgrades are prorated immediately. Retention offer may be provided once per account at agent discretion.

💬 Ask your question (or type 'exit'):  Human Escalation